# Week 3 Day 3 — GSW Filter and Single-Date Fit

This notebook tests `filter_bonds_for_fitting` on a realistic bond universe
and plots our fitted Svensson curve against the Fed's published zero yields
for a single date.

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

from termstructure.curves.svensson import (
    filter_bonds_for_fitting,
    fit_svensson,
    svensson_zero_rate,
)

## 1. Demonstrate the filter on a toy bond universe

We build a small DataFrame by hand — one valid bond plus one of each
excluded type — and verify that only the valid ones survive.

In [ ]:
universe = pd.DataFrame([
    dict(cusip='A', type='note',  maturity_years=5.0,  days_to_maturity=1825, callable=False, on_the_run=False, first_off_the_run=False, yield_pct=0.042),
    dict(cusip='B', type='note',  maturity_years=10.0, days_to_maturity=3650, callable=False, on_the_run=False, first_off_the_run=False, yield_pct=0.044),
    dict(cusip='C', type='tips',  maturity_years=10.0, days_to_maturity=3650, callable=False, on_the_run=False, first_off_the_run=False, yield_pct=0.018),  # excluded
    dict(cusip='D', type='bill',  maturity_years=0.5,  days_to_maturity=180,  callable=False, on_the_run=False, first_off_the_run=False, yield_pct=0.051),  # excluded
    dict(cusip='E', type='bond',  maturity_years=30.0, days_to_maturity=9000, callable=True,  on_the_run=False, first_off_the_run=False, yield_pct=0.046),  # excluded
    dict(cusip='F', type='note',  maturity_years=2.0,  days_to_maturity=60,   callable=False, on_the_run=False, first_off_the_run=False, yield_pct=0.048),  # excluded
    dict(cusip='G', type='note',  maturity_years=10.0, days_to_maturity=3650, callable=False, on_the_run=True,  first_off_the_run=False, yield_pct=0.039),  # excluded
    dict(cusip='H', type='note',  maturity_years=10.0, days_to_maturity=3650, callable=False, on_the_run=False, first_off_the_run=True,  yield_pct=0.040),  # excluded
])

filtered = filter_bonds_for_fitting(universe)

print(f'Original universe: {len(universe)} bonds')
print(f'After GSW filter:  {len(filtered)} bonds')
print()
print('Surviving bonds:')
filtered[['cusip', 'type', 'maturity_years', 'yield_pct']]

## 2. Single-date fit: our Svensson vs. the Fed's published zero yields

We use the Fed's daily zero yields (sveny01–sveny30) as input data,
fit our own Svensson to them, and plot the result.

Note: this is a circular check — we're fitting to already-fitted yields.
RMSE should be near-zero. When we later fit to raw bond prices,
RMSE will be larger and more meaningful.

In [ ]:
# Load the Fed parquet (built in Week 1 Day 3)
df = pd.read_parquet('../data/processed/treasury_bonds.parquet')
df['date'] = pd.to_datetime(df['date'])

row = df[df['date'] == '2024-01-02'].iloc[0]

maturities = np.arange(1, 31, dtype=float)
# Fed zero yields are in percent (e.g. 4.19); our functions use decimal (0.0419)
observed_yields = np.array([row[f'sveny{i:02d}'] for i in range(1, 31)]) / 100.0

print(f'Date: {row["date"].date()}')
print(f'1Y: {observed_yields[0]*100:.2f}%  |  10Y: {observed_yields[9]*100:.2f}%  |  30Y: {observed_yields[29]*100:.2f}%')

In [ ]:
params = fit_svensson(maturities, observed_yields)
b0, b1, b2, b3, l1, l2 = params

fitted_at_mats = svensson_zero_rate(maturities, *params)
residuals_bps = (fitted_at_mats - observed_yields) * 10_000
rmse_bps = np.sqrt(np.mean(residuals_bps**2))

print(f'β0={b0*100:.3f}%  β1={b1*100:.3f}%  β2={b2*100:.3f}%  β3={b3*100:.3f}%')
print(f'λ1={l1:.3f} yr  λ2={l2:.3f} yr')
print(f'RMSE: {rmse_bps:.3f} bps')

In [ ]:
tau_plot = np.linspace(0.25, 30, 500)
fitted_curve = svensson_zero_rate(tau_plot, *params)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 7), gridspec_kw={'height_ratios': [3, 1]})

ax1.plot(tau_plot, fitted_curve * 100, 'steelblue', lw=2, label='Our Svensson fit')
ax1.scatter(maturities, observed_yields * 100,
            color='black', zorder=5, s=30, label='Fed zero yields (input)')
ax1.set_ylabel('Zero Yield (%)')
ax1.set_title(f'Svensson Fit — 2024-01-02')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.bar(maturities, residuals_bps, color='steelblue', alpha=0.7)
ax2.axhline(0, color='black', lw=0.8)
ax2.set_xlabel('Maturity (years)')
ax2.set_ylabel('Residual (bp)')
ax2.set_title(f'Fit residuals  (RMSE = {rmse_bps:.3f} bp)')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('data/week3_day3_fit.png', dpi=150)
plt.show()

## What to expect

- **Top plot**: The curve should trace through the dots almost exactly (< 1 bp RMSE), because we're fitting to the Fed's already-smooth zero yields.
- **Bottom plot**: Residuals should be tiny and structureless. A systematic tilt (e.g., always positive at long end) means the optimizer is stuck in a local minimum — try different starting guesses.
- **Parameters**: β0 ≈ 30Y yield, β0+β1 ≈ 1Y yield, λ1 and λ2 locate the humps.